# 🚀 sim_mapping.sh 仿真建图系统使用指南

> **技术栈**: ROS 2 Humble · Gazebo Ignition Fortress 6 · Nav2 · Point-LIO · BehaviorTree.CPP v4  
> **机器人命名空间**: `red_standard_robot1`  
> **最后更新**: 2026年2月

---

## 📋 目录

| 编号 | 章节 | 内容 |
|:---:|------|------|
| 1 | [系统总览与架构](#1-系统总览与架构) | 三层启动架构、49个节点概览 |
| 2 | [第一层：sim_mapping.sh](#2-第一层sim_mappingsh-入口脚本) | Bash 入口、环境变量、GPU 渲染配置 |
| 3 | [第二层：launch_wrapper.py](#3-第二层launch_wrapperpy-编排器) | Python 编排器、进程生命周期管理 |
| 4 | [第三层A：Gazebo 仿真环境](#4-第三层agazebo-仿真环境) | 仿真器、机器人生成、ROS 桥接 |
| 5 | [第三层B：导航栈](#5-第三层b导航栈) | 点云转换 → LIO → 建图 → 规划 → 控制 |
| 6 | [完整数据流与 TF 链](#6-完整数据流与-tf-链) | 话题流向、TF 树、关键频率 |
| 7 | [快速启动 & 常见问题](#7-快速启动--常见问题) | 一键启动、环境变量速查、故障排查 |

<a id="1-系统总览与架构"></a>

## 1. 🏗️ 系统总览与架构

运行 `bash scripts/sim_mapping.sh` 后，系统按**三层结构**依次展开，最终启动约 **49 个 ROS 节点**：

```
┌─────────────────────────────────────────────────────────────────┐
│  第一层: sim_mapping.sh                                         │
│  ┌───────────────────────────────────────────────────────────┐  │
│  │ • 设置 GPU/渲染环境变量                                     │  │
│  │ • 定义 GAZEBO_CMD 和 SLAM_CMD                              │  │
│  │ • exec → launch_wrapper.py sim_mapping                    │  │
│  └───────────────────────────────────────────────────────────┘  │
│                            ↓                                    │
│  第二层: launch_wrapper.py (Python 编排器)                       │
│  ┌───────────────────────────────────────────────────────────┐  │
│  │ • 读取 nav2_params.yaml 配置                                │  │
│  │ • 后台启动 Gazebo ──(sleep 1s)──→ 前台启动 SLAM/Nav         │  │
│  │ • atexit 注册：前台退出时自动清理后台 Gazebo                  │  │
│  └───────────────────────────────────────────────────────────┘  │
│                     ↙                  ↘                        │
│  第三层A: Gazebo 仿真         第三层B: 导航栈                     │
│  ┌──────────────────┐    ┌──────────────────────────────┐      │
│  │ bringup_sim       │    │ rm_navigation_simulation     │      │
│  │ .launch.py        │    │ _launch.py                   │      │
│  │                   │    │                              │      │
│  │ • Gazebo 物理仿真  │    │ • ign_sim_pointcloud_tool   │      │
│  │ • 机器人 URDF 生成 │    │ • bringup (SLAM + Nav2)     │      │
│  │ • ROS↔Gz 桥接     │    │ • 手柄遥控                   │      │
│  │ • 裁判系统         │    │ • RViz 可视化                │      │
│  │                   │    │                              │      │
│  │ 约 12 个节点      │    │ 约 37 个节点                  │      │
│  └──────────────────┘    └──────────────────────────────────┘  │
└─────────────────────────────────────────────────────────────────┘
```

### 📊 关键数字

| 指标 | 值 |
|------|------|
| 总节点数 | ~49 |
| Gazebo 侧节点 | ~12（仿真器 + 桥接 + 裁判系统） |
| 导航侧节点 | ~37（SLAM + Nav2 全栈） |
| 核心参数文件 | `config/simulation/nav2_params.yaml`（~844 行） |
| 启动到就绪 | ~20 秒（含 Gazebo 渲染初始化 + IMU 初始化） |

<a id="2-第一层sim_mappingsh-入口脚本"></a>

## 2. 🐚 第一层：sim_mapping.sh 入口脚本

`scripts/sim_mapping.sh` 是整个系统的入口，它本身非常薄——只做三件事然后把控制权交给 Python。

### 2.1 环境变量设置

```bash
# ── GPU 渲染 ──
export __NV_PRIME_RENDER_OFFLOAD=1          # 强制使用独立 NVIDIA GPU
export __GLX_VENDOR_LIBRARY_NAME=nvidia     # GLX 渲染走 NVIDIA 驱动

# ── Gazebo 渲染引擎 ──
export IGN_GAZEBO_RENDER_ENGINE_SERVER=ogre2  # 服务端渲染引擎
export IGN_GAZEBO_RENDER_ENGINE_GUI=ogre2     # GUI 渲染引擎

# ── 界面 ──
export QT_FONT_DPI=120                       # 高 DPI 屏幕字体缩放
```

> ⚠️ 如果你用的是 Intel/AMD 集显，去掉 `__NV_PRIME_RENDER_OFFLOAD` 和 `__GLX_VENDOR_LIBRARY_NAME` 这两行即可。

### 2.2 定义核心启动命令

```bash
GAZEBO_CMD="ros2 launch rmu_gazebo_simulator bringup_sim.launch.py"
SLAM_CMD="ros2 launch pb2025_nav_bringup rm_navigation_simulation_launch.py slam:=True"
```

这两个变量分别定义了 **Gazebo 仿真环境** 和 **SLAM/导航栈** 的启动命令。你可以在运行前通过环境变量覆盖它们：

```bash
# 例：使用自定义参数文件
SLAM_CMD="ros2 launch pb2025_nav_bringup rm_navigation_simulation_launch.py slam:=True \
  params_file:=/path/to/my_params.yaml" bash scripts/sim_mapping.sh
```

### 2.3 交接给 Python

```bash
exec python3 "$SCRIPT_DIR/launch_wrapper.py" sim_mapping "$@"
```

`exec` 意味着**替换当前 shell 进程**——脚本不会返回到这一行之后，Python 直接接管 PID。`"$@"` 会把命令行额外参数透传下去。

<a id="3-第二层launch_wrapperpy-编排器"></a>

## 3. 🎛️ 第二层：launch_wrapper.py 编排器

`scripts/launch_wrapper.py` 是核心调度脚本，支持 4 种运行模式：

| 模式 | 命令 | 用途 |
|------|------|------|
| `sim_mapping` | `bash scripts/sim_mapping.sh` | 仿真建图（本文重点） |
| `sim_nav` | `bash scripts/sim_nav.sh` | 仿真导航（加载已有地图） |
| `reality_mapping` | `bash scripts/reality_mapping.sh` | 实车建图 |
| `reality_navigation` | `bash scripts/reality_nav.sh` | 实车导航 |

### 3.1 sim_mapping 模式的执行流程

```python
# launch_wrapper.py 第 608-614 行（简化）

# ① 后台启动 Gazebo
_launch_in_terminal(cfg, "Gazebo Sim", gazebo_cmd, background=True, bg=bg)

# ② 等待 1 秒让 Gazebo 开始初始化
time.sleep(1.0)

# ③ 前台启动 SLAM/导航栈（阻塞在这里直到 Ctrl+C）
_launch_in_terminal(cfg, "SLAM", slam_cmd, neupan_env)
```

### 3.2 进程生命周期管理

`launch_wrapper.py` 通过 `atexit` + `signal` 实现优雅退出：

```
用户按 Ctrl+C
    ↓
SIGINT 被捕获
    ↓
bg.cleanup() 被调用
    ↓
先发 SIGTERM 给 Gazebo 进程组 → 等 1 秒 → 再发 SIGKILL
    ↓
所有进程退出
```

> 🔑 **关键点**: 前台 SLAM 退出后，`atexit` 会自动清理后台 Gazebo。这意味着**不需要手动杀进程**。

### 3.3 终端模式

| 环境变量 | 行为 |
|----------|------|
| 不设置（默认） | 尝试用 `gnome-terminal` 打开新窗口 |
| `NO_NEW_TERMINAL=1` | 单终端模式：Gazebo 后台日志写文件，SLAM 前台输出 |
| Docker 容器内 | 自动切换为单终端模式 |

```bash
# 推荐在 VS Code 终端中使用单终端模式
NO_NEW_TERMINAL=1 bash scripts/sim_mapping.sh
```

### 3.4 自动配置读取

`launch_wrapper.py` 启动前会读取 `nav2_params.yaml` 中的配置来决定行为：

| 读取项 | 配置路径 | 作用 |
|--------|----------|------|
| 控制器插件 | `pb_navigation_switches.controller_plugin` | 若为 `neupan_nav2_controller` 则自动激活 NeuPAN 虚拟环境 |
| GT 里程计 | `pb_navigation_switches.enable_chassis_odometry_gt` | 控制是否桥接 Gazebo 底盘真值里程计 |
| 行为树 | `pb_navigation_switches.behavior_tree` | 打印行为树启用/禁用报告 |

<a id="4-第三层agazebo-仿真环境"></a>

## 4. 🌍 第三层A：Gazebo 仿真环境

`bringup_sim.launch.py` 负责启动整个仿真世界，包含 3 个子 launch：

```
bringup_sim.launch.py
├── gazebo.launch.py          ← 立即启动
├── spawn_robots.launch.py    ← 延迟 5 秒启动
└── referee_system.launch.py  ← 延迟 5 秒启动
```

> ⏱️ 延迟 5 秒是为了等待 Gazebo 渲染引擎完全初始化，避免 Ignition Fortress 6 已知的 `SceneManager::CreateVisual` 重复场景节点崩溃 bug。

### 4.1 各组件功能

#### 🎮 gazebo.launch.py — 仿真器核心

启动 Ignition Gazebo 物理仿真器，加载场景世界文件（如 `rmuc_2025_world.sdf`）。

- 世界选择由 `rmu_gazebo_simulator/config/gz_world.yaml` 控制
- 支持 `use_gui:=false` 无头模式（自动在无 DISPLAY 环境下启用）

#### 🤖 spawn_robots.launch.py — 机器人生成

为每个机器人（如 `red_standard_robot1`、`blue_standard_robot1`）执行：

| 节点 | 功能 |
|------|------|
| `robot_state_publisher` | 解析 URDF，发布 `base_footprint → chassis → gimbal_yaw` 等关节 TF |
| `ros_gz_bridge` | Gazebo ↔ ROS 话题桥接 |
| `robot_base` | 机器人底盘控制插件 |

**桥接的关键话题**:

| Gazebo 话题 | ROS 话题 | 内容 |
|-------------|----------|------|
| `/world/.../lidar` | `livox/lidar` | LiDAR 点云 (PointCloud2, ~13Hz) |
| `/world/.../imu` | `livox/imu` | IMU 数据 |
| `/world/.../joint_state` | `joint_states` | 关节状态 |
| — | `tf`, `tf_static` | URDF 坐标变换 |

#### 🏁 referee_system.launch.py — 裁判系统

模拟 RoboMaster 比赛裁判系统，提供 RFID、位姿等数据桥接。

<a id="5-第三层b导航栈"></a>

## 5. 🧭 第三层B：导航栈

`rm_navigation_simulation_launch.py` 是导航侧的顶层入口，启动 4 个子部分：

```
rm_navigation_simulation_launch.py
├── ign_sim_pointcloud_tool    ← 独立节点：点云格式转换
├── bringup_launch.py          ← 核心导航栈
│   ├── slam_launch.py         ← SLAM 通路 (slam:=True 时)
│   └── navigation_launch.py   ← Nav2 导航全栈
├── joy_teleop_launch.py       ← 手柄遥控
└── rviz_launch.py             ← RViz 可视化 (可选)
```

---

### 5.1 ign_sim_pointcloud_tool — 点云格式转换

| 项目 | 值 |
|------|------|
| **输入** | `livox/lidar` (Gazebo 原始 PointCloud2, ~13Hz) |
| **输出** | `velodyne_points` (添加 `ring` + `time` 字段) |
| **为什么需要** | Point-LIO 期望 Velodyne 格式的点云，需要逐点计算垂直角度并分配 ring ID |

关键参数（`nav2_params.yaml` 前 8 行）：
```yaml
ign_sim_pointcloud_tool:
  ros__parameters:
    pcd_topic: livox/lidar     # 订阅的原始点云话题
    n_scan: 32                 # 扫描线数
    horizon_scan: 1875         # 每线点数
    ang_bottom: 7.0            # 底部角度偏移
    ang_res_y: 1.84375         # 垂直角分辨率 = (7+52)/32
```

---

### 5.2 slam_launch.py — SLAM 通路

> ⚡ 仅在 `slam:=True` 时启用（建图模式）

| 节点 | 功能 | 关键输入 → 输出 |
|------|------|-----------------|
| **point_lio** | LiDAR-惯性里程计 | `velodyne_points` + `livox/imu` → `aft_mapped_to_init` + `cloud_registered` |
| **slam_toolbox** | 2D 栅格建图 | `obstacle_scan` → `/map` |
| **pointcloud_to_laserscan** | 3D→2D 投影 | `cloud_registered` → `obstacle_scan` (LaserScan) |
| **static_transform_publisher** | 静态 TF | 发布恒等变换 `map → odom` |
| **map_saver** | 地图保存 | 提供保存服务 |

#### Point-LIO 关键配置
```yaml
point_lio:
  ros__parameters:
    common:
      lid_topic: "velodyne_points"   # 点云输入
      imu_topic: "livox/imu"        # IMU 输入
    preprocess:
      lidar_type: 2                  # 2=Velodyne 格式
    publish:
      tf_send_en: False              # ⚠️ 不发布 TF（由 sensor_scan_generation 负责）
```

---

### 5.3 navigation_launch.py — Nav2 导航全栈

以 **composable node** 形式加载到 `nav2_container`（共享进程，减少通信开销）：

#### 🔄 里程计与坐标变换

| 节点 | 功能 | 发布的 TF |
|------|------|-----------|
| **loam_interface** | 转换 Point-LIO 输出为 Nav2 格式 | *(不发布 TF)* |
| **sensor_scan_generation** | 里程计融合 + 地形扫描 | ⭐ `odom → base_footprint` |
| **fake_vel_transform** | 云台补偿 + 速度变换 | `gimbal_yaw → gimbal_yaw_fake` (50Hz) |

#### 🗺️ 地形分析与代价地图

| 节点 | 输入 | 输出 | 用途 |
|------|------|------|------|
| **terrain_analysis** | `registered_scan` + `lidar_odometry` | `terrain_map` | local costmap 观测源 |
| **terrain_analysis_ext** | 同上 | `terrain_map_ext` | global costmap 观测源 |

> 💡 `terrain_map` 点云的 **intensity 字段 = 地面高度差 (disZ)**，`IntensityVoxelLayer` 根据 `min_obstacle_intensity` 阈值决定哪些点被标记为障碍物。

#### 🚗 路径规划与控制

| 节点 | 插件 | 功能 |
|------|------|------|
| **planner_server** | `ThetaStarPlanner` | 全局路径规划（A* 变体，支持任意角度） |
| **controller_server** | `OmniPidPursuitController` | 局部路径跟踪（全向 PID + 前瞻点追踪） |
| **smoother_server** | `SimpleSmoother` | 路径平滑 |
| **behavior_server** | Spin / BackUp / Wait | 恢复行为 |
| **bt_navigator** | BehaviorTree.CPP v4 | 行为树导航决策 |
| **velocity_smoother** | — | 速度限幅/滤波，输出最终 `cmd_vel` |
| **waypoint_follower** | — | 多航点依次执行 |

---

### 5.4 joy_teleop_launch.py — 手柄遥控

提供 Xbox/PS 手柄的底盘和云台控制，支持 `auto_control` / `manual_control` 模式切换。

<a id="6-完整数据流与-tf-链"></a>

## 6. 🔗 完整数据流与 TF 链

### 6.1 话题数据流

```
 ┌─────────────────── Gazebo Ignition ───────────────────┐
 │  livox/lidar (PointCloud2, ~13Hz)                     │
 │  livox/imu   (Imu, ~200Hz)                           │
 │  /model/.../joint_state → /joint_states               │
 └───────────────┬──────────────────────────────────────-─┘
                 │
                 ▼
   ┌─────────────────────────────┐
   │   ign_sim_pointcloud_tool   │  添加 ring + time 字段
   │   livox/lidar → velodyne_points (~12Hz)             │
   └──────────────┬──────────────┘
                  │
         ┌────────┴────────┐
         ▼                 ▼
   ┌──────────┐     ┌────────────────────────┐
   │ point_lio│     │ pointcloud_to_laserscan│
   │          │     │ cloud_registered →     │
   │ aft_mapped_to_init (~495Hz)             │
   │ cloud_registered   (~8Hz)               │
   └────┬─────┘     │ obstacle_scan (2D)     │
        │           └────────┬───────────────┘
        ▼                    ▼
   ┌──────────────┐   ┌──────────────┐
   │ loam_interface│   │ slam_toolbox │
   │              │   │ → /map       │
   │ lidar_odometry (~594Hz)         │
   │ registered_scan  (~10Hz)        │
   └───┬──────────┘   └─────────────-┘
       │
       ▼
   ┌───────────────────────────────┐
   │   sensor_scan_generation      │  ⭐ 发布 odom → base_footprint TF
   │   + terrain_analysis          │  → terrain_map (local costmap)
   │   + terrain_analysis_ext      │  → terrain_map_ext (global costmap)
   └───────────────┬───────────────┘
                   │
          ┌────────┴────────┐
          ▼                 ▼
   ┌──────────────┐  ┌──────────────────┐
   │ costmap_2d   │  │ planner_server   │
   │ (local/global)│  │ (ThetaStar)      │
   └──────┬───────┘  └────────┬─────────┘
          │                   │
          ▼                   ▼
   ┌──────────────────────────────────┐
   │   controller_server              │
   │   (OmniPidPursuitController)     │
   │   → cmd_vel_nav                  │
   └──────────────┬───────────────────┘
                  ▼
   ┌──────────────────────┐
   │   velocity_smoother  │  → cmd_vel (最终速度指令)
   └──────────────────────┘
```

### 6.2 TF 树结构

```
map                          ← static_transform_publisher (恒等变换)
 └── odom                   ← sensor_scan_generation
      └── base_footprint    ← sensor_scan_generation (里程计位姿)
           └── chassis      ← robot_state_publisher (URDF)
                └── gimbal_yaw          ← Gazebo joint_states
                     └── gimbal_yaw_fake ← fake_vel_transform (50Hz)
                          └── livox_frame ← robot_state_publisher (URDF)
```

### 6.3 关键频率参考

| 话题 / TF | 频率 | 来源节点 |
|-----------|------|---------|
| `livox/lidar` | ~13 Hz | Gazebo → IGN Bridge |
| `velodyne_points` | ~12 Hz | ign_sim_pointcloud_tool |
| `aft_mapped_to_init` | ~495 Hz | point_lio |
| `cloud_registered` | ~8 Hz | point_lio |
| `lidar_odometry` | ~594 Hz | loam_interface |
| `registered_scan` | ~10 Hz | loam_interface |
| `odom → base_footprint` TF | ~600 Hz | sensor_scan_generation |
| `gimbal_yaw → gimbal_yaw_fake` TF | 50 Hz | fake_vel_transform |
| `terrain_map` | ~10 Hz | terrain_analysis |
| `obstacle_scan` | ~8 Hz | pointcloud_to_laserscan |

> ⚠️ 以上频率为仿真实测参考值，实际值受 CPU/GPU 负载影响。

<a id="7-快速启动--常见问题"></a>

## 7. 🚀 快速启动 & 常见问题

### 7.1 快速启动

```bash
# 1. 仿真建图模式
cd ~/rm_code/2026_1_21/ros2_ws
bash scripts/sim_mapping.sh

# 2. 仿真导航模式（需要先有地图）
bash scripts/sim_nav.sh

# 3. 不弹出新终端窗口（所有输出合并到当前终端）
NO_NEW_TERMINAL=1 bash scripts/sim_mapping.sh

# 4. 仅编译改动过的包（快速增量编译）
bash scripts/quick_build.sh
```

### 7.2 环境变量参考

| 变量 | 默认值 | 说明 |
|------|--------|------|
| `NO_NEW_TERMINAL` | *(未设置)* | 设为 `1` 禁用多终端弹窗，所有进程输出合并 |
| `__NV_PRIME_RENDER_OFFLOAD` | `1` | NVIDIA GPU 渲染卸载 |
| `__GLX_VENDOR_LIBRARY_NAME` | `nvidia` | 指定 GLX 使用 NVIDIA 驱动 |
| `IGN_GAZEBO_RESOURCE_PATH` | *(自动设置)* | Gazebo 模型/世界资源搜索路径 |
| `GAZEBO_CMD` | *(脚本内定义)* | Gazebo 仿真启动命令 |
| `SLAM_CMD` | *(脚本内定义)* | SLAM 导航栈启动命令 |

### 7.3 常见问题排查

#### ❌ TF 树断裂 / map 找不到 transform

**症状**：RViz 显示 `No transform from [base_footprint] to [map]`

**排查步骤**：
1. 检查所有节点是否正常运行：`ros2 node list | wc -l`（预期 ~49 个节点）
2. 检查 `point_lio` 是否收到点云：`ros2 topic hz /velodyne_points`
3. 检查 TF 是否完整：`ros2 run tf2_tools view_frames`
4. 最常见原因：部分进程被意外 kill → **重新运行 `sim_mapping.sh`**

#### ❌ 代价地图过度膨胀

**症状**：Costmap 中大面积标记为障碍物，路径规划失败

**排查步骤**：
1. 检查 `terrain_map` 点云 intensity 分布：
   ```bash
   ros2 topic echo /terrain_map --field intensity --once
   ```
2. 关键参数（`nav2_params.yaml`）：
   - `min_obstacle_intensity`: 低于此值的点被忽略（当前 **0.25**）
   - `inflation_radius`: 障碍物膨胀半径（local: **0.35m**, global: **0.40m**）
   - `cost_scaling_factor`: 膨胀衰减速率（当前 **8.0**，越大衰减越快）

#### ❌ Gazebo 启动后黑屏 / 崩溃

**排查步骤**：
1. 确认 GPU 驱动正常：`nvidia-smi`
2. 确认 Gazebo 版本：`ign gazebo --version`（需要 Fortress 6.x）
3. 检查资源路径：`echo $IGN_GAZEBO_RESOURCE_PATH`
4. 手动测试 Gazebo：`ign gazebo empty.sdf`

#### ❌ 手柄遥控无响应

**排查步骤**：
1. 确认手柄设备：`ls /dev/input/js*`
2. 检查 joy 节点：`ros2 node list | grep joy`
3. 检查话题输出：`ros2 topic echo /joy --once`
4. 确认处于 `manual_control` 模式

---

### 7.4 关键配置文件索引

| 文件 | 路径 | 内容 |
|------|------|------|
| **nav2_params.yaml** | `pb2025_nav_bringup/config/simulation/` | 所有导航节点参数 (844行) |
| **launch_wrapper.py** | `scripts/` | 启动编排器 (645行) |
| **sim_mapping.sh** | `scripts/` | 仿真建图入口 |
| **sim_nav.sh** | `scripts/` | 仿真导航入口 |
| **bringup_sim.launch.py** | `pb2025_nav_bringup/launch/` | Gazebo 环境启动 |
| **rm_navigation_simulation_launch.py** | `pb2025_nav_bringup/launch/` | 导航栈启动 |
| **slam_launch.py** | `pb2025_nav_bringup/launch/` | SLAM 节点组 |
| **navigation_launch.py** | `pb2025_nav_bringup/launch/` | Nav2 节点组 |

---

> 📝 **文档版本**：2025-02-16 | 基于 `ros2_ws` 工作空间实际代码分析生成